# Talanta P2 time-of-day bake on the Kaggle GPU (internal, private)

One run bakes one lightmap set (or a part of it, ONLY) of the web_hero_v2 twin at one time of day, in one Blender process, and renders that time's panorama.
Scene: blender/out/frozen_FL.blend of the render pack (assets/render_pack, 2026-10-26: frozen_FK plus the seat re-check stand detail). Same groups, UVs and file layout as the 07:30 set (assets/web_hero_v2_bake).
Order and test: blender/kaggle/BAKE_ON_KAGGLE.md.

How to run:
- Settings: Accelerator GPU T4 x2, Internet on, input the private dataset talanta-render-pack (the 2026-10-26 pack version).
- Set the variables in the first code cell. "Save Version", then "Save & Run All (Commit)", private. The maps stay as the version output (lightmaps/<HHMM>/).
- Download the output, then stop. Do not leave an idle session open.

Traffic in the maps is Visualization. The output is internal only. Do not publish this notebook or its output.

In [ ]:
# Settings of this run
HHMM = "1030"                 # 1030, 1805 or 2000
PRESET = {"1030": "daylight_artistic", "1805": "sunset_artistic", "2000": "night"}[HHMM]
# 2000: the manifest sun "night" (20:30 EAT, venue lights on; the sky is dark from about 19:00, so the light is the 20:00 light)
ONLY = "stadium_shell,stadium_fins,stadium_ornament,roof"   # the GPU test (CPU sub-run 1a); "" bakes every group and writes bake_log.json
SAMPLES = 128
PANORAMA = False              # True: also the 8192 x 4096 panorama of this time (after a full bake)
print(HHMM, PRESET, ONLY or "all groups")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2; free -g | head -2

In [ ]:
# Blender and the pack go to /tmp, so that the version output holds the frames only.
import glob, json, os, shutil, subprocess, time
WORK = "/tmp/talanta"
BLENDER_URL = "https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz"  # same as the GPU test
BLENDER = f"{WORK}/blender/blender"
t_setup = time.time()
if not os.path.exists(BLENDER):
    os.makedirs(f"{WORK}/blender", exist_ok=True)
    subprocess.run(f"wget -q {BLENDER_URL} -O /tmp/b.tar.xz && tar -xf /tmp/b.tar.xz -C {WORK}/blender --strip-components=1 "
                   "&& rm /tmp/b.tar.xz", shell=True, check=True)
subprocess.run([BLENDER, "-b", "--version"], check=True)

# The pack root holds blender/ and blender/out/frozen_FL.blend (pack 2026-10-26; frozen_FK.blend in pack 7e19362). The input is read-only, so copy it.
PACK = f"{WORK}/pack/talanta_render_pack"
if not os.path.exists(PACK):
    os.makedirs(f"{WORK}/pack", exist_ok=True)
    unpacked = glob.glob("/kaggle/input/**/talanta_render_pack/blender/out/frozen_F*.blend", recursive=True)
    if unpacked:
        shutil.copytree(unpacked[0].rsplit("/blender/", 1)[0], PACK)
    else:
        parts = sorted(q for q in glob.glob("/kaggle/input/**/talanta_render_pack.tar*", recursive=True) if not q.endswith(".sha256"))
        print(parts)
        subprocess.run(f"cat {' '.join(parts)} | tar -x -C {WORK}/pack", shell=True, check=True)
print(os.listdir(PACK))

In [ ]:
# The inline gpu_pre.py of the GPU test notebook (same as blender/kaggle/gpu_pre.py in the repo).
GPU_PRE = f"{WORK}/gpu_pre.py"
open(GPU_PRE, "w").write('''prefs = bpy.context.preferences.addons["cycles"].preferences
chosen = None
for kind in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = kind
    except TypeError:
        continue
    prefs.get_devices()
    if any(d.type == kind for d in prefs.devices):
        chosen = kind
        break
if chosen is None:
    raise SystemExit("gpu_pre.py: no OptiX or CUDA device found")
for d in prefs.devices:
    d.use = d.type == chosen
s.cycles.device = "GPU"
print("gpu_pre.py:", chosen, [d.name for d in prefs.devices if d.use])
''')
print("setup", round(time.time() - t_setup), "s")

In [ ]:
# The bake: one Blender process; maps, stages.json (light scale and seconds per batch) and, for a full run, bake_log.json
# and talanta_web_baked.blend go to /kaggle/working/lightmaps/<HHMM>/ (the version output)
import threading
OUT = f"/kaggle/working/lightmaps/{HHMM}"
os.makedirs(f"{OUT}/maps", exist_ok=True)
env = dict(os.environ, TALANTA_SITE_BLEND=f"{PACK}/blender/out/frozen_FL.blend", TALANTA_PRE=GPU_PRE,
           TALANTA_BAKE_OUT=OUT, TALANTA_BAKE_PRESET=PRESET, PYTHONUNBUFFERED="1")
gpu_mem_max = {}
def _watch():
    while not _stop.is_set():
        q = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True).stdout
        for line in q.strip().splitlines():
            i, used, total = [x.strip() for x in line.split(",")]
            gpu_mem_max[i] = [max(int(used), gpu_mem_max.get(i, [0])[0]), int(total)]
        _stop.wait(10)
_stop = threading.Event(); threading.Thread(target=_watch, daemon=True).start()
args = ([f"only={ONLY}"] if ONLY else []) + [str(SAMPLES)]
cmd = [BLENDER, "-b", "--factory-startup", "--python-exit-code", "1", "--python", f"{PACK}/blender/web_hero/bake_lightmaps.py", "--", *args]
print(" ".join(cmd)); t0 = time.time()
try:
    r = subprocess.run(cmd, cwd=PACK, env=env, capture_output=True, text=True)
finally:
    _stop.set()
open(f"{OUT}/bake_run.log", "w").write(r.stdout[-200000:] + r.stderr[-50000:])
print("\n".join(l for l in r.stdout.splitlines() if l.startswith(("bake ", "light scales", "gpu_pre"))))
print("bake", round((time.time() - t0) / 3600, 2), "h, rc", r.returncode)
assert r.returncode == 0, "bake failed: see bake_run.log in the output"

In [ ]:
# The panorama of this time (only after a full bake)
if PANORAMA:
    cmd = [BLENDER, "-b", "--factory-startup", "--python-exit-code", "1", "--python", f"{PACK}/blender/web_hero/render_panorama.py", "--",
           str(SAMPLES), "8192", f"preset={PRESET}", "near=800", f"out={OUT}/env"]
    t0 = time.time(); r = subprocess.run(cmd, cwd=PACK, env=env, capture_output=True, text=True)
    open(f"{OUT}/pano_run.log", "w").write(r.stdout[-100000:] + r.stderr[-20000:])
    print("panorama", round((time.time() - t0) / 60, 1), "min, rc", r.returncode); assert r.returncode == 0

In [ ]:
# Run record (it goes with the maps to assets/web_hero_v2_bake_<HHMM>)
import hashlib
st = json.load(open(f"{OUT}/maps/stages.json"))
files = sorted(glob.glob(f"{OUT}/**/*.*", recursive=True))
rec = {"hhmm": HHMM, "preset": PRESET, "only": ONLY or "all", "samples": SAMPLES, "stage_key": st.get("key"),
       "batch_seconds": {k: v["seconds"] for k, v in st["done"].items() if isinstance(v, dict) and "seconds" in v},
       "light_scales": {k: v.get("light_scale") for k, v in st["done"].items() if isinstance(v, dict) and "light_scale" in v},
       "gpu_memory_max_mib": gpu_mem_max,
       "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.split("\n")[:2],
       "sha256": {os.path.relpath(f, OUT): hashlib.sha256(open(f, "rb").read()).hexdigest() for f in files if not f.endswith((".log", "run_record.json"))},
       "claim": "Internal only. Traffic is Visualization."}
json.dump(rec, open(f"{OUT}/run_record.json", "w"), indent=1)
print(json.dumps({k: v for k, v in rec.items() if k != "sha256"}, indent=1))